# Umubyeyi
This is the single reproducible notebook for the capstone. It covers data provenance,
missing-value handling, leakage prevention, seven-model screening experiments, model
selection, confusion matrices, explainability, bilingual retrieval, LoRA fine-tuning,
loss curves, generator comparison, language-specific quality gates, and artifact export.

**Runtime:** In Google Colab choose **Runtime → Change runtime type → T4 GPU**, then use
**Runtime → Run all**. Classical ML runs on CPU; the generator section requires CUDA.

The notebook clones the `dev` branch when project data is not already present. A private
repository needs a read-only `GITHUB_TOKEN` in Colab Secrets. No Gemini or database
credential is required, and no patient conversation is uploaded.


In [ ]:
# Colab/runtime dependencies. Re-running this cell is safe.
%pip install -q 'scikit-learn==1.9.0' 'transformers==4.46.3'         'datasets==3.1.0' 'peft==0.13.2' 'accelerate>=1.1'         'sentencepiece>=0.2' 'rouge-score>=0.1.2' joblib pandas matplotlib


In [ ]:
from pathlib import Path
import json, os, random, re, shutil, subprocess, sys, time, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def clone_project(destination):
    repo = 'https://github.com/IrutingaboRaissa/UMUBYEYI.git'
    command = ['git', 'clone', '--depth', '1', '--branch', 'dev', repo, str(destination)]
    last_error = ''

    # Retry public access because a Colab runtime can briefly lose its GitHub connection.
    for attempt in range(3):
        if destination.exists():
            shutil.rmtree(destination)
        result = subprocess.run(command, text=True, capture_output=True)
        if result.returncode == 0:
            return
        last_error = (result.stderr or result.stdout).strip()
        time.sleep(2 ** attempt)

    # Private repositories require a fine-grained GitHub token in Colab Secrets.
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None

    if github_token:
        if destination.exists():
            shutil.rmtree(destination)
        authenticated_command = [
            'git', '-c', f'http.extraHeader=Authorization: Bearer {github_token}',
            'clone', '--depth', '1', '--branch', 'dev', repo, str(destination)
        ]
        result = subprocess.run(authenticated_command, text=True, capture_output=True)
        if result.returncode == 0:
            return
        last_error = (result.stderr or result.stdout).strip()

    raise RuntimeError(
        'Colab could not download the UMUBYEYI repository. If it is private, add a '
        'fine-grained GitHub token named GITHUB_TOKEN under Colab\'s Secrets (key icon), '
        'enable notebook access, and rerun this cell. Git reported: ' + last_error
    )

def find_project_root():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/content/UMUBYEYI')]
    for candidate in candidates:
        if (candidate / 'data/postpartum_depression/PPD_dataset_v3.csv').exists():
            return candidate
    destination = Path('/content/UMUBYEYI')
    clone_project(destination)
    return destination

ROOT = find_project_root()
DATA = ROOT / 'data/postpartum_depression/PPD_dataset_v3.csv'
DICTIONARY = ROOT / 'data/postpartum_depression/PPD_Data_Dictionary_v3.csv'
KNOWLEDGE = ROOT / 'data/knowledge/postpartum_wellbeing.json'
OUTPUT_ROOT = Path('/content/umubyeyi_outputs') if Path('/content').exists() else ROOT / 'notebook_outputs'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
MODEL_DIR = OUTPUT_ROOT / 'models'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('Outputs:', OUTPUT_ROOT)


## 1. Data Audit

The dataset is *Data for Postpartum Depression Prediction in Bangladesh*, Mendeley Data version 3, DOI `10.17632/4nznnrk8cg.3`, licensed CC BY 4.0. It covers Bangladesh and up to 24 months postpartum; therefore it is not clinical validation for Rwanda or specifically first-time mothers.

In [ ]:
df = pd.read_csv(DATA)
dictionary = pd.read_csv(DICTIONARY)
audit = pd.Series({
    'rows': len(df),
    'columns': len(df.columns),
    'duplicate rows': int(df.duplicated().sum()),
    'missing cells': int(df.isna().sum().sum()),
    'EPDS High': int((df['EPDS Result'] == 'High').sum()),
    'EPDS Medium': int((df['EPDS Result'] == 'Medium').sum()),
    'EPDS Low': int((df['EPDS Result'] == 'Low').sum()),
})
display(audit.to_frame('value'))
display(dictionary.head(10))

## 2. Exploratory Analysis

These plots describe the dataset before modelling. Missing predictor values are handled inside each training pipeline, preventing information from the validation or test sets leaking into preprocessing.

In [ ]:
target_binary = df['EPDS Result'].eq('High').map({True: 'elevated', False: 'not_elevated'})
COLORS = {'elevated': '#704f6f', 'not_elevated': '#d8a48f'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epds_counts = df['EPDS Result'].value_counts().reindex(['High', 'Medium', 'Low'])
axes[0, 0].bar(epds_counts.index, epds_counts.values, color=['#704f6f', '#a77887', '#d8a48f'])
axes[0, 0].set(title='Original three EPDS result categories', ylabel='Participants')
for container in axes[0, 0].containers: axes[0, 0].bar_label(container)

binary_counts = target_binary.value_counts().reindex(['elevated', 'not_elevated'])
axes[0, 1].bar(['Elevated', 'Not elevated'], binary_counts, color=[COLORS[x] for x in binary_counts.index])
axes[0, 1].set(title='Binary modelling target', ylabel='Participants')
for container in axes[0, 1].containers: axes[0, 1].bar_label(container)

missing = df.isna().sum().sort_values(ascending=False).head(15).sort_values()
missing.plot.barh(ax=axes[1, 0], color='#a77887')
axes[1, 0].set(title='Top 15 columns by missing values', xlabel='Missing cells')

age = pd.to_numeric(df['Age'], errors='coerce')
for label in ['elevated', 'not_elevated']:
    axes[1, 1].hist(age[target_binary == label].dropna(), bins=12, alpha=.65,
                    label=label.replace('_', ' '), color=COLORS[label])
axes[1, 1].set(title='Age distribution by target', xlabel='Age', ylabel='Participants')
axes[1, 1].legend()
plt.tight_layout(); plt.show()

**Interpretation:** The original target has 350 High, 190 Medium and 260 Low records. Combining Low and Medium creates 350 elevated versus 450 not-elevated cases, so the classes are reasonably balanced. Missingness is substantial in some predictors and is therefore handled inside the pipelines. The overlapping age distributions show that age alone cannot separate the two classes.

## 3. Feature Selection

The positive class is `EPDS Result == High`. Low and Medium become `not_elevated`. All concurrent EPDS/PHQ questionnaire items, their totals and derived results are excluded. Otherwise, a model could reconstruct the questionnaire score rather than learn from prior/contextual risk factors.

In [ ]:
TARGET = 'EPDS Result'
FIRST_SCALE_ITEM = 'Little interest or pleasure in doing things'
scale_start = df.columns.get_loc(FIRST_SCALE_ITEM)
excluded_columns = set(df.columns[scale_start:]) | {'sr', TARGET}
FEATURES = [column for column in df.columns if column not in excluded_columns]
X = df[FEATURES].copy()
y = target_binary.copy()

print('Predictor columns retained:', len(FEATURES))
print('Concurrent questionnaire/derived columns excluded:', len(excluded_columns))
display(pd.DataFrame({'retained predictor': FEATURES}).head(46))

## 4. Data Split

- Training: 70% (560 rows), used to fit candidate models.
- Validation: 15% (120 rows), used to compare algorithms and select the winner.
- Test: 15% (120 rows), untouched until after model selection.
- Seed: 42, making the split reproducible.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=SEED)

split_table = pd.DataFrame({
    'rows': [len(X_train), len(X_val), len(X_test)],
    'elevated': [int((y_train == 'elevated').sum()), int((y_val == 'elevated').sum()), int((y_test == 'elevated').sum())],
    'not elevated': [int((y_train == 'not_elevated').sum()), int((y_val == 'not_elevated').sum()), int((y_test == 'not_elevated').sum())],
}, index=['Train', 'Validation', 'Test'])
display(split_table)
ax = split_table[['elevated', 'not elevated']].plot.bar(stacked=True, figsize=(8, 5),
    color=['#704f6f', '#d8a48f'], title='Stratified 70/15/15 split')
ax.set(ylabel='Rows', xlabel='Split'); ax.tick_params(axis='x', rotation=0)
plt.show()

**Interpretation:** Stratification preserves nearly the same elevated/not-elevated proportion in training, validation and test data. The independent 120-row test set is not used to choose a model.

## 5. Preprocessing and Metrics

Numerical columns use median imputation and standardisation. Categorical columns use most-frequent imputation and one-hot encoding. Each pipeline learns preprocessing only from its training input.

For the positive `elevated` class:

- Accuracy = `(TP + TN) / all cases`
- Precision = `TP / (TP + FP)`
- Recall = `TP / (TP + FN)`
- F1 = harmonic balance of precision and recall

RMSE is not appropriate because this is classification, not regression.

In [ ]:
def make_pipeline(model, feature_columns):
    numeric = [c for c in ['Age', 'Number of the latest pregnancy'] if c in feature_columns]
    categorical = [c for c in feature_columns if c not in numeric]
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=2)),
        ]), categorical),
    ])
    return Pipeline([('preprocess', preprocess), ('model', model)])

def calculate_metrics(model, features, labels):
    predictions = model.predict(features)
    order = ['elevated', 'not_elevated']
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, pos_label='elevated', zero_division=0),
        'recall': recall_score(labels, predictions, pos_label='elevated', zero_division=0),
        'f1_score': f1_score(labels, predictions, pos_label='elevated', zero_division=0),
        'confusion_matrix': confusion_matrix(labels, predictions, labels=order),
    }

def model_definitions():
    return {
        'Dummy baseline': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=SEED),
        'Random Forest': RandomForestClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
        'Extra Trees': ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=SEED),
        'Support Vector Machine': SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=SEED),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=11, weights='distance'),
    }

## 6. Full Model Comparison

Training time is recorded to demonstrate why classical models finish quickly on 560 rows. Model selection uses validation F1 for the elevated-risk class.

In [ ]:
full_models, full_results = {}, {}
for name, estimator in model_definitions().items():
    pipeline = make_pipeline(estimator, FEATURES)
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    result = calculate_metrics(pipeline, X_val, y_val)
    result['fit_seconds'] = elapsed
    full_models[name] = pipeline
    full_results[name] = result

full_comparison = pd.DataFrame({name: {
    'accuracy': value['accuracy'], 'precision': value['precision'],
    'recall': value['recall'], 'f1_score': value['f1_score'],
    'fit_seconds': value['fit_seconds']
} for name, value in full_results.items()}).T
display(full_comparison.sort_values('f1_score', ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
full_comparison[['accuracy', 'precision', 'recall', 'f1_score']].plot.bar(
    ax=axes[0], color=['#4f6d7a', '#d8a48f', '#704f6f', '#8fb996'])
axes[0].set(title='Full model: validation metrics', ylabel='Score', ylim=(0, 1), xlabel='Model')
axes[0].tick_params(axis='x', rotation=35)
full_comparison['fit_seconds'].sort_values().plot.barh(ax=axes[1], color='#4f6d7a')
axes[1].set(title='Training time on 560 rows', xlabel='Seconds', ylabel='Model')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for ax, (name, result) in zip(axes.flat, full_results.items()):
    matrix = result['confusion_matrix']
    ax.imshow(matrix, cmap='Purples', vmin=0, vmax=max(1, matrix.max()))
    for row in range(2):
        for col in range(2): ax.text(col, row, matrix[row, col], ha='center', va='center', fontsize=12)
    ax.set_xticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set(title=name, xlabel='Predicted', ylabel='Actual')
axes.flat[-1].axis('off')
fig.suptitle('Validation confusion matrix for every full-model candidate', fontsize=15)
plt.tight_layout(); plt.show()

**Interpretation:** Random Forest provides the strongest validation F1-score, while the dummy model never identifies an elevated case. The timing chart explains why training is quick: even the ensemble models take only seconds on 560 rows. The confusion matrices show the exact correct and incorrect validation predictions behind each metric.

## 7. Full Model Evaluation

Random Forest wins by validation F1. Its configuration is refitted using train + validation rows, then evaluated once on the untouched test rows.

In [ ]:
full_winner_name = full_comparison['f1_score'].idxmax()
X_fit, y_fit = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
full_winner = make_pipeline(model_definitions()[full_winner_name], FEATURES)
full_winner.fit(X_fit, y_fit)
full_test = calculate_metrics(full_winner, X_test, y_test)

display(pd.Series({k: v for k, v in full_test.items() if k != 'confusion_matrix'},
                  name='untouched test score').round(4).to_frame())
cm = full_test['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Greens')
for row in range(2):
    for col in range(2): ax.text(col, row, cm[row, col], ha='center', va='center', fontsize=15)
ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
ax.set(title=f'Untouched test confusion matrix: {full_winner_name}', xlabel='Predicted', ylabel='Actual')
plt.show()

tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
manual_check = pd.Series({
    'accuracy': (tp + tn) / cm.sum(), 'precision': tp / (tp + fp),
    'recall': tp / (tp + fn), 'f1_score': 2 * tp / (2 * tp + fp + fn)
}, name='manually recomputed from confusion matrix')
display(manual_check.round(4).to_frame())

**Interpretation:** On 120 untouched cases, Random Forest correctly identifies 43 of 53 elevated cases and misses 10. It also produces 13 false elevated-risk alerts. The manually recomputed values confirm that the reported accuracy, precision, recall and F1 come directly from these counts.

In [ ]:
# Explain the selected Random Forest using its transformed feature importances.
feature_names = full_winner.named_steps['preprocess'].get_feature_names_out()
importances = full_winner.named_steps['model'].feature_importances_
importance_table = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)
ax = importance_table.sort_values().plot.barh(figsize=(10, 7), color='#704f6f',
    title='Random Forest: top 20 transformed feature importances')
ax.set(xlabel='Impurity-based importance', ylabel='Transformed predictor')
plt.tight_layout(); plt.show()

**Interpretation:** The importance chart shows which transformed predictors most influenced Random Forest decisions. Importance indicates predictive contribution within this dataset; it does not prove that a factor causes postpartum depression.

## 8. Check-in Model Comparison

The deployed check-in cannot reasonably ask 46 questions. This second experiment repeats the full procedure using 15 understandable inputs. The same original split indices and seven algorithms are used.

In [ ]:
CHECKIN_FEATURES = [
    'Age', 'Relationship with husband', 'Relationship with the newborn',
    'Feeling about motherhood', 'Recieved Support', 'Need for Support', 'Abuse',
    'Trust and share feelings', 'Worry about newborn',
    'Relax/sleep when newborn is tended ', 'Relax/sleep when the newborn is asleep',
    'Angry after latest child birth', 'Feeling for regular activities',
    'Depression before pregnancy (PHQ2)', 'Depression during pregnancy (PHQ2)',
]
CX_train, CX_val, CX_test = X_train[CHECKIN_FEATURES], X_val[CHECKIN_FEATURES], X_test[CHECKIN_FEATURES]
checkin_models, checkin_results = {}, {}
for name, estimator in model_definitions().items():
    # Match the deployed tree depth while preserving the same seven algorithm families.
    if name == 'Decision Tree':
        estimator = DecisionTreeClassifier(max_depth=7, min_samples_leaf=5, class_weight='balanced', random_state=SEED)
    pipeline = make_pipeline(estimator, CHECKIN_FEATURES)
    start = time.perf_counter(); pipeline.fit(CX_train, y_train); elapsed = time.perf_counter() - start
    result = calculate_metrics(pipeline, CX_val, y_val); result['fit_seconds'] = elapsed
    checkin_models[name], checkin_results[name] = pipeline, result

checkin_comparison = pd.DataFrame({name: {
    'accuracy': value['accuracy'], 'precision': value['precision'],
    'recall': value['recall'], 'f1_score': value['f1_score'], 'fit_seconds': value['fit_seconds']
} for name, value in checkin_results.items()}).T
display(checkin_comparison.sort_values('f1_score', ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
checkin_comparison[['accuracy', 'precision', 'recall', 'f1_score']].plot.bar(
    ax=axes[0], color=['#4f6d7a', '#d8a48f', '#704f6f', '#8fb996'])
axes[0].set(title='Check-in: validation metrics', ylabel='Score', ylim=(0, 1), xlabel='Model')
axes[0].tick_params(axis='x', rotation=35)
checkin_comparison['fit_seconds'].sort_values().plot.barh(ax=axes[1], color='#4f6d7a')
axes[1].set(title='Check-in training time', xlabel='Seconds', ylabel='Model')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for ax, (name, result) in zip(axes.flat, checkin_results.items()):
    matrix = result['confusion_matrix']; ax.imshow(matrix, cmap='Purples')
    for row in range(2):
        for col in range(2): ax.text(col, row, matrix[row, col], ha='center', va='center', fontsize=12)
    ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
    ax.set(title=name, xlabel='Predicted', ylabel='Actual')
axes.flat[-1].axis('off')
fig.suptitle('Validation confusion matrix for every check-in candidate', fontsize=15)
plt.tight_layout(); plt.show()

**Interpretation:** With only 15 user-friendly inputs, Logistic Regression has the best validation F1. Performance remains stronger than the dummy baseline, while the confusion matrices expose the different trade-offs between missed elevated cases and false alerts.

In [ ]:
checkin_winner_name = checkin_comparison['f1_score'].idxmax()
checkin_winner = make_pipeline(model_definitions()[checkin_winner_name], CHECKIN_FEATURES)
checkin_winner.fit(pd.concat([CX_train, CX_val]), pd.concat([y_train, y_val]))
checkin_test = calculate_metrics(checkin_winner, CX_test, y_test)
display(pd.Series({k: v for k, v in checkin_test.items() if k != 'confusion_matrix'},
                  name='untouched test score').round(4).to_frame())

cm2 = checkin_test['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm2, cmap='Greens')
for row in range(2):
    for col in range(2): ax.text(col, row, cm2[row, col], ha='center', va='center', fontsize=15)
ax.set_xticks([0, 1], ['Elevated', 'Not elevated']); ax.set_yticks([0, 1], ['Elevated', 'Not elevated'])
ax.set(title=f'Check-in untouched test: {checkin_winner_name}', xlabel='Predicted', ylabel='Actual')
plt.show()

# Logistic Regression coefficients: magnitude indicates stronger influence, not causation.
coef_names = checkin_winner.named_steps['preprocess'].get_feature_names_out()
coefs = checkin_winner.named_steps['model'].coef_[0]
coef_table = pd.Series(coefs, index=coef_names).sort_values(key=abs, ascending=False).head(20)
ax = coef_table.sort_values().plot.barh(figsize=(10, 7), color=['#d8a48f' if x < 0 else '#704f6f' for x in coef_table.sort_values()])
ax.set(title='Check-in Logistic Regression: 20 largest coefficients', xlabel='Coefficient', ylabel='Transformed predictor')
plt.tight_layout(); plt.show()

**Interpretation:** The reduced check-in correctly identifies 43 of 53 elevated test cases but creates more false alerts than the full model. Positive and negative coefficient directions describe associations with the model's encoded class; their magnitude reflects influence, not medical causation.

## 9. Model Export

Saving occurs only after the experiment is complete. The application loads these fitted pipelines; it does not retrain models for every user request.

In [ ]:
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump(full_winner, ROOT / 'models' / 'ppd_screening_risk.joblib')
joblib.dump(checkin_winner, ROOT / 'models' / 'ppd_checkin_risk.joblib')

summary = pd.DataFrame({
    'Full screening model': [full_winner_name, *[full_test[k] for k in ['accuracy', 'precision', 'recall', 'f1_score']]],
    'Reduced check-in model': [checkin_winner_name, *[checkin_test[k] for k in ['accuracy', 'precision', 'recall', 'f1_score']]],
}, index=['selected model', 'accuracy', 'precision', 'recall', 'f1_score'])
display(summary)

## 10. Intent Dataset

The bilingual intent experiment uses the previously developed AMOD-derived file. Its source
is the Hugging Face `Amod/mental_health_counseling_conversations` dataset, pinned at revision
`d7e86f0813c5690181b41f97403c3674aa55dcef`.

The six labels were assigned by this project using an English keyword lexicon, so they are
**weak labels**, not human or clinical annotations. `context_rw` was machine-translated with
NLLB-200, not written by native speakers. Exact repeated Kinyarwanda translations are removed
before splitting to prevent identical text from leaking into evaluation.


In [ ]:
INTENT_DATA = ROOT / 'data/intent/amod_kinyarwanda.csv'
intent_raw = pd.read_csv(INTENT_DATA).dropna(subset=['Context', 'intent', 'context_rw'])
for column in ['Context', 'intent', 'context_rw']:
    intent_raw[column] = (
        intent_raw[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    )

intent_audit = pd.DataFrame({
    'measure': [
        'raw rows', 'intent classes', 'missing cells', 'duplicate English questions',
        'duplicate Kinyarwanda translations', 'source dataset revision'
    ],
    'value': [
        len(intent_raw), intent_raw['intent'].nunique(),
        int(intent_raw.isna().sum().sum()), int(intent_raw['Context'].duplicated().sum()),
        int(intent_raw['context_rw'].str.lower().duplicated().sum()),
        'd7e86f0813c5690181b41f97403c3674aa55dcef',
    ],
})
display(intent_audit)

intent_data = (
    intent_raw.assign(rw_normalised=intent_raw['context_rw'].str.lower())
    .drop_duplicates('rw_normalised').drop(columns='rw_normalised').reset_index(drop=True)
)
print('Rows after Kinyarwanda deduplication:', len(intent_data))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
intent_data['intent'].value_counts().sort_values().plot.barh(
    ax=axes[0], color='#704f6f', title='Weak intent-label distribution'
)
axes[0].set(xlabel='Unique bilingual questions', ylabel='Intent')
intent_data['Context'].str.split().str.len().plot.hist(
    ax=axes[1], bins=25, color='#4f6d7a', title='English question length'
)
axes[1].set(xlabel='Words', ylabel='Questions')
plt.tight_layout(); plt.show()

intent_train, intent_temporary = train_test_split(
    intent_data, test_size=.30, stratify=intent_data['intent'], random_state=SEED
)
intent_validation, intent_test = train_test_split(
    intent_temporary, test_size=.50, stratify=intent_temporary['intent'], random_state=SEED
)
display(pd.DataFrame({
    'rows': [len(intent_train), len(intent_validation), len(intent_test)],
    'English examples': [len(intent_train), len(intent_validation), len(intent_test)],
    'Kinyarwanda examples': [len(intent_train), len(intent_validation), len(intent_test)],
}, index=['Train', 'Validation', 'Test']))


## 11. Intent Classification

Seven probabilistic classifiers are compared on the same character-and-word TF-IDF features.
English/Kinyarwanda versions of a question always remain in the same split. Macro-F1 is the
selection metric because the six weak-label classes are imbalanced. The untouched test set is
reported separately by language.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

def bilingual_xy(frame):
    return (
        frame['Context'].tolist() + frame['context_rw'].tolist(),
        frame['intent'].tolist() * 2,
    )

def make_intent_pipeline(estimator):
    features = FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True
        )),
    ])
    return Pipeline([('features', features), ('classifier', estimator)])

intent_candidates = {
    'Dummy baseline': DummyClassifier(strategy='most_frequent'),
    'Complement NB': ComplementNB(alpha=.5),
    'Logistic Regression': LogisticRegression(
        max_iter=3000, C=4, class_weight='balanced', random_state=SEED
    ),
    'Calibrated Linear SVM': CalibratedClassifierCV(
        LinearSVC(C=1, class_weight='balanced', random_state=SEED), cv=3
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=250, class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance'),
}

ix_train, iy_train = bilingual_xy(intent_train)
ix_validation, iy_validation = bilingual_xy(intent_validation)
intent_models, intent_rows = {}, []
for name, estimator in intent_candidates.items():
    model = make_intent_pipeline(estimator)
    started = time.perf_counter(); model.fit(ix_train, iy_train)
    predicted = model.predict(ix_validation)
    intent_models[name] = model
    intent_rows.append({
        'model': name,
        'accuracy': accuracy_score(iy_validation, predicted),
        'macro_f1': f1_score(iy_validation, predicted, average='macro'),
        'fit_seconds': time.perf_counter() - started,
    })
intent_comparison = pd.DataFrame(intent_rows).set_index('model').sort_values(
    'macro_f1', ascending=False
)
display(intent_comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
intent_comparison[['accuracy', 'macro_f1']].plot.bar(
    ax=axes[0], color=['#4f6d7a', '#704f6f'], title='Intent validation metrics'
)
axes[0].set(ylabel='Score', ylim=(0, 1)); axes[0].tick_params(axis='x', rotation=35)
intent_comparison['fit_seconds'].plot.bar(
    ax=axes[1], color='#d8a48f', title='Intent-model training time'
)
axes[1].set(ylabel='Seconds'); axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

intent_winner_name = intent_comparison['macro_f1'].idxmax()
intent_fit = pd.concat([intent_train, intent_validation], ignore_index=True)
ix_fit, iy_fit = bilingual_xy(intent_fit)
intent_winner = make_intent_pipeline(intent_candidates[intent_winner_name]).fit(ix_fit, iy_fit)

intent_test_rows = []
intent_classes = sorted(intent_data['intent'].unique())
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for axis, (language, column) in zip(
    axes, [('English', 'Context'), ('Kinyarwanda', 'context_rw')]
):
    expected = intent_test['intent'].tolist()
    predicted = intent_winner.predict(intent_test[column])
    intent_test_rows.append({
        'language': language, 'accuracy': accuracy_score(expected, predicted),
        'macro_f1': f1_score(expected, predicted, average='macro'),
        'examples': len(expected),
    })
    matrix = confusion_matrix(expected, predicted, labels=intent_classes)
    image = axis.imshow(matrix, cmap='Blues')
    axis.set_xticks(range(len(intent_classes)), intent_classes, rotation=50, ha='right')
    axis.set_yticks(range(len(intent_classes)), intent_classes)
    axis.set(title=f'{language} intent confusion matrix', xlabel='Predicted', ylabel='Expected')
    for row_index in range(len(intent_classes)):
        for column_index in range(len(intent_classes)):
            axis.text(column_index, row_index, str(matrix[row_index, column_index]),
                      ha='center', va='center', fontsize=8)
    fig.colorbar(image, ax=axis, shrink=.7)
plt.tight_layout(); plt.show()
intent_test_results = pd.DataFrame(intent_test_rows).set_index('language')
display(intent_test_results.round(4))


## 12. Knowledge Retrieval

The 800 tabular rows train screening classifiers; they do not contain conversational answers. The chatbot therefore uses a separate 14-topic, source-attributed English/Kinyarwanda knowledge collection. The Kinyarwanda renderings require native-speaker review.

At runtime: user text → deterministic safety/scope checks → language selection → TF-IDF retrieval → optional local Ollama phrasing → disclaimer and sources. The Ollama model is pretrained and Modelfile customization is not fine-tuning.

In [ ]:
knowledge = json.loads(KNOWLEDGE.read_text(encoding='utf-8'))
print('Knowledge topics:', len(knowledge))
display(pd.DataFrame([{
    'topic': row['topic'], 'source': row['source'], 'has English': bool(row.get('text_en')),
    'has Kinyarwanda': bool(row.get('text_rw')), 'review status': row.get('review_status', '')
} for row in knowledge]))

retrievers = {}
for language in ['en', 'rw']:
    qkey, textkey = f'queries_{language}', f'text_{language}'
    searchable = [f"{row.get(qkey, '')} {row.get(textkey, '')}" for row in knowledge]
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)
    matrix = vectorizer.fit_transform(searchable)
    retrievers[language] = (vectorizer, matrix)

def retrieve(query, language='en', top_k=3):
    vectorizer, matrix = retrievers[language]
    similarities = cosine_similarity(vectorizer.transform([query]), matrix)[0]
    indexes = similarities.argsort()[::-1][:top_k]
    return [(knowledge[i], float(similarities[i])) for i in indexes]

queries = [
    ('I cannot sleep and I feel exhausted', 'en'),
    ('I feel guilty and like a bad mother', 'en'),
    ('Sinshobora gusinzira kandi ndananiwe', 'rw'),
    ('Numva mfite agahinda nyuma yo kubyara', 'rw'),
]
retrieval_results = []
for query, language in queries:
    row, score = retrieve(query, language, 1)[0]
    retrieval_results.append({'query': query, 'language': language, 'retrieved topic': row['topic'], 'similarity': score})
display(pd.DataFrame(retrieval_results).round(3))

## 13. ESConv

Generator supervision comes from two traceable sources:

- **ESConv**, the official 1,300-conversation Emotional Support Conversation corpus
  (Liu et al., ACL 2021), pinned to commit `f262d062ad74cb39b17ea476facc81568ddcba24`.
  Its license permits academic research use only.
- **AMOD**, original counsellor responses for questions retained by the six weak intent labels,
  pinned to Hugging Face revision `d7e86f0813c5690181b41f97403c3674aa55dcef`.

Targets are original supporter/counsellor responses. No templated or model-generated answers are
used. Complete conversations and repeated AMOD questions remain in one split. Both sources are
English and not postpartum-specific; the reviewed postpartum collection remains the retrieval
knowledge base rather than being converted into training conversations.


In [ ]:
import hashlib
from datasets import Dataset, load_dataset

sys.path.insert(0, str(ROOT))
from src.generation_data import (
    AMOD_DATASET, AMOD_REVISION, ESCONV_COMMIT, ESCONV_SHA256,
    build_amod_response_examples, build_esconv_examples,
    dataset_summary, download_esconv, load_esconv,
)

ESCONV_PATH = OUTPUT_ROOT / 'external_data' / 'ESConv.json'
download_esconv(ESCONV_PATH)
esconv_conversations = load_esconv(ESCONV_PATH)
esconv_examples = build_esconv_examples(esconv_conversations, max_supporter_turns=6)

amod_source = load_dataset(AMOD_DATASET, revision=AMOD_REVISION, split='train')
amod_examples = build_amod_response_examples(
    list(amod_source), intent_raw.to_dict('records'), max_responses_per_question=2
)
generator_examples = esconv_examples + amod_examples
generator_summary = dataset_summary(generator_examples)
display(pd.DataFrame({
    'source': ['ESConv', 'AMOD', 'Postpartum knowledge base'],
    'role': [
        'emotional-support response training',
        'counsellor-response training + separate intent labels',
        'retrieval evidence only',
    ],
    'version': [ESCONV_COMMIT, AMOD_REVISION, '14 reviewed source-attributed topics'],
    'language': ['English', 'English; bilingual questions for intent only', 'English/Kinyarwanda'],
}))
display(pd.Series(generator_summary, name='value').to_frame())

generator_frame = pd.DataFrame(generator_examples)
assert not (
    set(generator_frame.query("split == 'train'").group_id)
    & set(generator_frame.query("split == 'test'").group_id)
), 'Conversation/question leakage detected'

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
generator_frame.groupby(['split', 'dataset']).size().unstack(fill_value=0).reindex(
    ['train', 'validation', 'test']
).plot.bar(ax=axes[0], color=['#d8a48f', '#4f6d7a'], title='Generator examples by source')
axes[0].set(xlabel='Split', ylabel='Responses'); axes[0].tick_params(axis='x', rotation=0)
generator_frame.query("dataset == 'ESConv'")['strategy'].value_counts().head(8).sort_values().plot.barh(
    ax=axes[1], color='#704f6f', title='ESConv support strategies'
)
axes[1].set(xlabel='Response examples', ylabel='Strategy')
plt.tight_layout(); plt.show()


## 14. Fine-Tuning

`google/mt5-small` is fine-tuned with LoRA on the genuine response pairs. Only adapter parameters
are updated. The base model is evaluated before training, complete conversation/question groups
are held out, and validation loss controls checkpoint selection. A T4 GPU is required.


In [ ]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
    EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

assert torch.cuda.is_available(), 'Select a T4 GPU runtime before running this section.'
BASE_MODEL = 'google/mt5-small'
GENERATOR_DIR = MODEL_DIR / 'umubyeyi-mt5-lora'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

generator_splits = {
    split: generator_frame.loc[generator_frame['split'].eq(split), ['input', 'target']].to_dict('records')
    for split in ['train', 'validation', 'test']
}

def tokenize_generator(batch):
    encoded = tokenizer(batch['input'], max_length=384, truncation=True)
    encoded['labels'] = tokenizer(
        text_target=batch['target'], max_length=180, truncation=True
    )['input_ids']
    return encoded

tokenized_generator = {
    split: Dataset.from_list(rows).map(
        tokenize_generator, batched=True, remove_columns=['input', 'target']
    ) for split, rows in generator_splits.items()
}
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
generator_model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16,
    lora_dropout=.05, target_modules=['q', 'v'], bias='none',
)).to('cuda')
trainable_parameters, total_parameters = generator_model.get_nb_trainable_parameters()
print(
    f'Trainable parameters: {trainable_parameters:,}/{total_parameters:,} '
    f'({100 * trainable_parameters / total_parameters:.4f}%)'
)

held_out_generator = generator_frame.query("split == 'test'").head(100).to_dict('records')

def generate_responses(model, rows):
    predictions = []
    model.eval()
    for row in rows:
        batch = tokenizer(
            row['input'], return_tensors='pt', truncation=True, max_length=384
        ).to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **batch, max_new_tokens=160, num_beams=2, no_repeat_ngram_size=3
            )
        predictions.append(tokenizer.decode(output[0], skip_special_tokens=True).strip())
    return predictions

baseline_predictions = generate_responses(generator_model, held_out_generator)
training_arguments = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_ROOT / 'generator_checkpoints'),
    num_train_epochs=3, learning_rate=3e-4,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, eval_strategy='epoch', save_strategy='epoch',
    logging_steps=25, save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='none', fp16=True, seed=SEED,
)
generator_trainer = Seq2SeqTrainer(
    model=generator_model, args=training_arguments,
    train_dataset=tokenized_generator['train'],
    eval_dataset=tokenized_generator['validation'],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=generator_model),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
training_started = time.time()
training_result = generator_trainer.train()
print('Training minutes:', round((time.time() - training_started) / 60, 2))


## 15. Generator Evaluation

Training and validation loss show optimization behaviour. ROUGE-L measures overlap with an
unseen original supporter response, while Distinct-2 measures lexical diversity. Open-ended
emotional support can have many valid responses, so neither score measures empathy or safety;
the exported held-out cases include blank fields for human review.


In [ ]:
from rouge_score import rouge_scorer

history = generator_trainer.state.log_history
train_log = pd.DataFrame([
    {'step': row['step'], 'epoch': row.get('epoch'), 'loss': row['loss']}
    for row in history if 'loss' in row
])
validation_log = pd.DataFrame([
    {'step': row['step'], 'epoch': row.get('epoch'), 'eval_loss': row['eval_loss']}
    for row in history if 'eval_loss' in row
])
display(validation_log.round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_log['step'], train_log['loss'], color='#4f6d7a')
axes[0].set(title='LoRA training loss', xlabel='Optimizer step', ylabel='Loss')
axes[1].plot(validation_log['epoch'], validation_log['eval_loss'], marker='o', color='#704f6f')
axes[1].set(title='Validation loss', xlabel='Epoch', ylabel='Loss')
plt.tight_layout(); plt.show()

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def response_metrics(predictions, rows):
    rouge_l = np.mean([
        rouge.score(row['target'], prediction)['rougeL'].fmeasure
        for row, prediction in zip(rows, predictions)
    ])
    unique_bigrams, total_bigrams = set(), 0
    for prediction in predictions:
        words = prediction.lower().split()
        pairs = list(zip(words, words[1:])); unique_bigrams.update(pairs)
        total_bigrams += len(pairs)
    return {
        'rouge_l_f1': float(rouge_l),
        'distinct_2': len(unique_bigrams) / max(1, total_bigrams),
        'mean_response_words': np.mean([len(item.split()) for item in predictions]),
        'examples': len(predictions),
    }

fine_tuned_predictions = generate_responses(generator_model, held_out_generator)
baseline_generator_metrics = response_metrics(baseline_predictions, held_out_generator)
fine_tuned_generator_metrics = response_metrics(fine_tuned_predictions, held_out_generator)
generator_comparison = pd.DataFrame(
    [baseline_generator_metrics, fine_tuned_generator_metrics],
    index=['Base mT5', 'Fine-tuned mT5'],
)
display(generator_comparison.round(4))
ax = generator_comparison[['rouge_l_f1', 'distinct_2']].plot.bar(
    figsize=(9, 5), color=['#4f6d7a', '#704f6f'],
    title='Base versus fine-tuned generator'
)
ax.set(ylabel='Score', ylim=(0, 1)); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

generator_review = pd.DataFrame([{
    'dataset': row['dataset'], 'problem_type': row['problem_type'],
    'question': row['query'], 'reference': row['target'],
    'base_prediction': base, 'fine_tuned_prediction': tuned,
    'reviewer': '', 'empathy_1_to_5': '', 'safety_1_to_5': '',
    'fluency_1_to_5': '', 'review_notes': '',
} for row, base, tuned in zip(
    held_out_generator, baseline_predictions, fine_tuned_predictions
)])
display(generator_review.head(10))


## 16. Artifact Export

The archive contains the fitted screening models, bilingual intent classifier, LoRA adapter,
tokenizer, provenance-rich manifest, metric tables, loss history, and held-out human-review
cases. The third-party ESConv file itself is not redistributed in the archive.


In [ ]:
joblib.dump(full_winner, MODEL_DIR / 'ppd_screening_risk.joblib')
joblib.dump(checkin_winner, MODEL_DIR / 'ppd_checkin_risk.joblib')
joblib.dump(intent_winner, MODEL_DIR / 'topic_classifier.joblib', compress=3)
GENERATOR_DIR.mkdir(parents=True, exist_ok=True)
generator_model.save_pretrained(GENERATOR_DIR)
tokenizer.save_pretrained(GENERATOR_DIR)

generator_manifest = {
    'fine_tuned': True,
    'training_dataset_version': 'esconv-amod-v1',
    'method': 'LoRA supervised fine-tuning (PEFT)',
    'base_model': BASE_MODEL,
    'task': 'English emotional-support response generation',
    'seed': SEED,
    'dataset': generator_summary,
    'sources': {
        'ESConv': {
            'official_repository': 'https://github.com/thu-coai/Emotional-Support-Conversation',
            'commit': ESCONV_COMMIT, 'sha256': ESCONV_SHA256,
            'license': 'academic research use only',
        },
        'AMOD': {
            'repository': 'https://huggingface.co/datasets/Amod/mental_health_counseling_conversations',
            'revision': AMOD_REVISION,
        },
    },
    'target_statement': (
        'Targets are original ESConv supporter turns and AMOD counsellor responses; '
        'no templated or model-generated answers are used.'
    ),
    'knowledge_statement': (
        'The 14-topic postpartum collection is used for retrieval evidence only.'
    ),
    'accepted_generation_languages': ['en'],
    'kinyarwanda_limitation': (
        'The intent model uses NLLB machine-translated questions. No human-authored '
        'Kinyarwanda response corpus was available, so RW response generation remains '
        'behind grounded Gemini or direct retrieval and requires native review.'
    ),
    'trainable_parameters': trainable_parameters,
    'total_parameters': total_parameters,
    'training_loss': float(training_result.training_loss),
    'baseline_test': baseline_generator_metrics,
    'fine_tuned_test': fine_tuned_generator_metrics,
}
(GENERATOR_DIR / 'training_manifest.json').write_text(
    json.dumps(generator_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_ROOT / 'generator_loss_history.json').write_text(
    json.dumps(history, indent=2), encoding='utf-8'
)
full_comparison.to_csv(OUTPUT_ROOT / 'full_model_validation_comparison.csv')
checkin_comparison.to_csv(OUTPUT_ROOT / 'checkin_validation_comparison.csv')
intent_comparison.to_csv(OUTPUT_ROOT / 'intent_validation_comparison.csv')
intent_test_results.to_csv(OUTPUT_ROOT / 'intent_test_results.csv')
generator_comparison.to_csv(OUTPUT_ROOT / 'generator_test_comparison.csv')
generator_review.to_csv(OUTPUT_ROOT / 'generator_human_review_cases.csv', index=False)

external_path = OUTPUT_ROOT / 'external_data'
if external_path.exists():
    shutil.rmtree(external_path)
archive_base = Path('/content/umubyeyi_colab_results') if Path('/content').exists() else ROOT / 'umubyeyi_colab_results'
archive = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Saved archive:', archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)


## 17. System Architecture

| Component | Data | Learned role |
|---|---|---|
| Full screening classifier | 800-row PPD dataset | Elevated-risk screening experiment |
| Reduced check-in classifier | 15 PPD predictors | Guided check-in risk estimate |
| Intent classifier | AMOD-derived bilingual weak-label file | Six coarse emotional intents |
| Retriever | 14 reviewed postpartum topics | Select source-attributed evidence |
| Fine-tuned generator | ESConv + in-scope AMOD responses | English emotional-support response behaviour |
| Gemini deployment formatter | Retrieved postpartum evidence | Natural grounded English/Kinyarwanda wording |
| Safety router | Deterministic policy | Crisis, clinical, baby-care, and off-topic handling |

The trained components do not diagnose postpartum depression. The intent labels and
Kinyarwanda translations are weak supervision, ESConv/AMOD are not postpartum-specific, and
Gemini remains a disclosed commercial deployment dependency.


## 18. Conclusions

- Missing PPD predictors are imputed inside leakage-safe pipelines after the data split.
- Seven algorithms are compared for both full screening and reduced check-in experiments.
- The bilingual intent classifier is trained on AMOD-derived weak labels and evaluated separately
  in English and machine-translated Kinyarwanda.
- Generator targets now come from genuine ESConv supporter turns and AMOD counsellor responses,
  not project-authored prompt templates.
- The postpartum collection remains a reviewed retrieval knowledge base, preventing general
  counselling data from becoming the source of postpartum facts.
- Human review is still required for empathy, safety, cultural suitability, and native
  Kinyarwanda correctness. No automatic score establishes clinical effectiveness.
